# FinRL-DeepSeek — Multi-Seed Ablation Study A→F (Colab Pro)

**GPU accelerator must be ON** (A100 or V100 recommended for multi-seed).
This notebook runs the full ablation study over multiple random seeds to compute the mean and standard deviation of all metrics, which is required for rigorous NeurIPS submissions.

| Config | Algorithm | M1 Confidence | M2 Reward Shaping | M3 Circuit Breaker |
|--------|-----------|:---:|:---:|:---:|
| A | PPO baseline | — | — | — |
| B | PPO + LLM | — | — | — |
| C | CPPO baseline | — | — | — |
| D | CPPO + Confidence | ✓ | — | — |
| E | CPPO + Shaping | ✓ | ✓ | — |
| F | Full model | ✓ | ✓ | ✓ |

**Estimated runtime for 5 seeds on A100: ~10–15 h total**

## 1. Install dependencies

In [ ]:
%%capture
!pip install gymnasium yfinance stockstats openai datasets huggingface_hub PyYAML python-dotenv websockets

## 2. Clone repository

In [ ]:
import os
REPO = "https://github.com/testS7ven/FinRL_DeepSeek.git"
!git clone {REPO} /content/FinRL_DeepSeek
os.chdir("/content/FinRL_DeepSeek")
print("Working directory:", os.getcwd())

## 3. Check GPU

In [ ]:
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 4. GPU-patched training functions

We monkey-patch `ppo_train` and `cppo_train` to move the agent and tensors to GPU.

In [ ]:
import sys
sys.path.insert(0, "/content/FinRL_DeepSeek")

import time
import numpy as np
import torch
from pathlib import Path

from risk_first.training.networks import ActorCritic
from risk_first.training.buffer import PPOBuffer, CPPOBuffer

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def ppo_train_gpu(env, cfg, save_dir="risk_first/models", run_name="ppo", log_path=None):
    torch.manual_seed(cfg.get("seed", 42))
    np.random.seed(cfg.get("seed", 42))
    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.shape[0]
    hidden  = cfg.get("hidden_sizes", [512, 512])
    agent = ActorCritic(obs_dim, act_dim, hidden).to(DEVICE)
    pi_opt = torch.optim.Adam(list(agent.pi_net.parameters()) + [agent.log_std], lr=cfg["pi_lr"])
    v_opt = torch.optim.Adam(agent.v_net.parameters(), lr=cfg["vf_lr"])
    buf = PPOBuffer(obs_dim, act_dim, size=cfg["steps_per_epoch"], gamma=cfg["gamma"], lam=cfg.get("lam", 0.97))
    save_path = Path(save_dir)
    save_path.mkdir(parents=True, exist_ok=True)
    logs = []
    last_portfolio_history = []
    obs, _ = env.reset()
    ep_ret = 0.0

    for epoch in range(cfg["epochs"]):
        t0 = time.time()
        ep_rets = []
        portfolio_hist = []
        for step in range(cfg["steps_per_epoch"]):
            with torch.no_grad():
                obs_t = torch.as_tensor(obs, dtype=torch.float32).to(DEVICE)
                dist, v = agent(obs_t)
                a = dist.sample()
                a_clipped = torch.clamp(a, -1.0, 1.0)
                logp = dist.log_prob(a_clipped).sum()
                act_np = a_clipped.cpu().numpy()
                val = float(v.cpu())
                logp_val = float(logp.cpu())
            next_obs, rew, done, _, info = env.step(act_np)
            buf.store(obs, act_np, rew, val, logp_val)
            portfolio_hist.append(info.get("portfolio_value", 0.0))
            ep_ret += rew
            obs = next_obs
            timeout = (step == cfg["steps_per_epoch"] - 1)
            if done or timeout:
                if done:
                    ep_rets.append(ep_ret)
                    buf.finish_path(last_val=0.0)
                else:
                    with torch.no_grad():
                        obs_t2 = torch.as_tensor(obs, dtype=torch.float32).to(DEVICE)
                        _, lv = agent(obs_t2)
                        last_val = float(lv.cpu())
                    buf.finish_path(last_val=last_val)
                obs, _ = env.reset()
                ep_ret = 0.0
        last_portfolio_history = portfolio_hist
        data = buf.get()
        obs_t    = torch.as_tensor(data["obs"],  dtype=torch.float32).to(DEVICE)
        act_t    = torch.as_tensor(data["act"],  dtype=torch.float32).to(DEVICE)
        adv_t    = torch.as_tensor(data["adv"],  dtype=torch.float32).to(DEVICE)
        ret_t    = torch.as_tensor(data["ret"],  dtype=torch.float32).to(DEVICE)
        logp_old = torch.as_tensor(data["logp"], dtype=torch.float32).to(DEVICE)
        kl = 0.0
        for _ in range(cfg["train_pi_iters"]):
            pi_opt.zero_grad()
            logp_new, _, _ = agent.evaluate(obs_t, act_t)
            ratio    = torch.exp(logp_new - logp_old)
            clip_adv = torch.clamp(ratio, 1 - cfg["clip_ratio"], 1 + cfg["clip_ratio"]) * adv_t
            pi_loss  = -torch.min(ratio * adv_t, clip_adv).mean()
            kl = float((logp_old - logp_new).detach().mean())
            if abs(kl) > 1.5 * cfg["target_kl"]:
                break
            pi_loss.backward()
            pi_opt.step()
        for _ in range(cfg["train_v_iters"]):
            v_opt.zero_grad()
            _, v_new, _ = agent.evaluate(obs_t, act_t)
            v_loss = ((v_new - ret_t) ** 2).mean()
            v_loss.backward()
            v_opt.step()
        avg_ret = float(np.mean(ep_rets)) if ep_rets else 0.0
        line = (f"[{run_name}] Epoch {epoch+1:3d}/{cfg['epochs']} | "
                f"AvgEpRet {avg_ret:12.2f} | KL {kl:.4f} | Time {time.time()-t0:.1f}s")
        # print(line) # Suppressed to keep multiseed output clean
        logs.append(line)
    torch.save(agent.state_dict(), save_path / f"{run_name}_agent.pth")
    if log_path:
        Path(log_path).parent.mkdir(parents=True, exist_ok=True)
        Path(log_path).write_text("\n".join(logs))
    print(f"[{run_name}] Saved to {save_path}/{run_name}_agent.pth")
    return last_portfolio_history

def cppo_train_gpu(env, cfg, cppo_cfg, save_dir="risk_first/models", run_name="cppo", log_path=None):
    torch.manual_seed(cfg.get("seed", 42))
    np.random.seed(cfg.get("seed", 42))
    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.shape[0]
    hidden  = cfg.get("hidden_sizes", [512, 512])
    agent = ActorCritic(obs_dim, act_dim, hidden).to(DEVICE)
    pi_opt = torch.optim.Adam(list(agent.pi_net.parameters()) + [agent.log_std], lr=cfg["pi_lr"])
    v_opt = torch.optim.Adam(agent.v_net.parameters(), lr=cfg["vf_lr"])
    buf = CPPOBuffer(obs_dim, act_dim, size=cfg["steps_per_epoch"], gamma=cfg["gamma"], lam=cfg.get("lam", 0.97))
    alpha, beta, cvar_clip, delay = cppo_cfg["alpha"], cppo_cfg["beta"], cppo_cfg["cvar_clip_ratio"], cppo_cfg["delay"]
    nu, cvarlam = 0.0, 0.0
    save_path = Path(save_dir)
    save_path.mkdir(parents=True, exist_ok=True)
    logs = []
    last_portfolio_history = []
    ep_rets_for_cvar = []
    obs, _ = env.reset()
    ep_ret = 0.0

    for epoch in range(cfg["epochs"]):
        t0 = time.time()
        ep_rets = []
        portfolio_hist = []
        for step in range(cfg["steps_per_epoch"]):
            with torch.no_grad():
                obs_t = torch.as_tensor(obs, dtype=torch.float32).to(DEVICE)
                dist, v = agent(obs_t)
                a = dist.sample()
                a_clipped = torch.clamp(a, -1.0, 1.0)
                logp = dist.log_prob(a_clipped).sum()
                act_np = a_clipped.cpu().numpy()
                val = float(v.cpu())
                logp_val = float(logp.cpu())
            next_obs, rew, done, _, info = env.step(act_np)
            llm_risk_factor = info.get("llm_risk_factor", 1.0)
            adjusted_D = llm_risk_factor * (ep_ret + val - nu)
            if adjusted_D < nu and cvarlam > 0:
                valupdate = delay * cvarlam / (1.0 - alpha) * (nu - adjusted_D)
                valupdate = min(valupdate, abs(val) * cvar_clip)
            else:
                valupdate = 0.0
            buf.store(obs, act_np, rew, val, logp_val, valupdate)
            portfolio_hist.append(info.get("portfolio_value", 0.0))
            ep_ret += rew
            obs = next_obs
            timeout = (step == cfg["steps_per_epoch"] - 1)
            if done or timeout:
                if done:
                    ep_rets.append(ep_ret)
                    ep_rets_for_cvar.append(ep_ret)
                    buf.finish_path(last_val=0.0)
                else:
                    with torch.no_grad():
                        obs_t2 = torch.as_tensor(obs, dtype=torch.float32).to(DEVICE)
                        _, lv = agent(obs_t2)
                        last_val = float(lv.cpu())
                    buf.finish_path(last_val=last_val)
                obs, _ = env.reset()
                ep_ret = 0.0
        last_portfolio_history = portfolio_hist
        if len(ep_rets_for_cvar) >= 2:
            sorted_rets = np.sort(ep_rets_for_cvar)
            tail_idx = max(1, min(int(len(sorted_rets) * (1.0 - alpha)), len(sorted_rets) - 1))
            nu   = float(sorted_rets[tail_idx])
            cvar = -float(sorted_rets[:tail_idx].mean())
            if cvar > beta:
                cvarlam = max(0.0, cvarlam + 0.1 * (cvar - beta))
            else:
                cvarlam = max(0.0, cvarlam - 0.05)
            ep_rets_for_cvar.clear()
        data = buf.get()
        obs_t    = torch.as_tensor(data["obs"],  dtype=torch.float32).to(DEVICE)
        act_t    = torch.as_tensor(data["act"],  dtype=torch.float32).to(DEVICE)
        adv_t    = torch.as_tensor(data["adv"],  dtype=torch.float32).to(DEVICE)
        ret_t    = torch.as_tensor(data["ret"],  dtype=torch.float32).to(DEVICE)
        logp_old = torch.as_tensor(data["logp"], dtype=torch.float32).to(DEVICE)
        kl = 0.0
        for _ in range(cfg["train_pi_iters"]):
            pi_opt.zero_grad()
            logp_new, _, _ = agent.evaluate(obs_t, act_t)
            ratio    = torch.exp(logp_new - logp_old)
            clip_adv = torch.clamp(ratio, 1 - cfg["clip_ratio"], 1 + cfg["clip_ratio"]) * adv_t
            pi_loss  = -torch.min(ratio * adv_t, clip_adv).mean()
            kl = float((logp_old - logp_new).detach().mean())
            if abs(kl) > 1.5 * cfg["target_kl"]:
                break
            pi_loss.backward()
            pi_opt.step()
        for _ in range(cfg["train_v_iters"]):
            v_opt.zero_grad()
            _, v_new, _ = agent.evaluate(obs_t, act_t)
            v_loss = ((v_new - ret_t) ** 2).mean()
            v_loss.backward()
            v_opt.step()
        avg_ret = float(np.mean(ep_rets)) if ep_rets else 0.0
        line = (f"[{run_name}] Epoch {epoch+1:3d}/{cfg['epochs']} | "
                f"AvgEpRet {avg_ret:12.2f} | KL {kl:.4f} | "
                f"nu {nu:.1f} | cvarlam {cvarlam:.4f} | Time {time.time()-t0:.1f}s")
        # print(line) # Suppressed to keep multiseed output clean
        logs.append(line)
    torch.save(agent.state_dict(), save_path / f"{run_name}_agent.pth")
    if log_path:
        Path(log_path).parent.mkdir(parents=True, exist_ok=True)
        Path(log_path).write_text("\n".join(logs))
    print(f"[{run_name}] Saved to {save_path}/{run_name}_agent.pth")
    return last_portfolio_history

print("GPU-patched trainers ready.")

## 5. Monkey-patch pipeline & run ablation

In [ ]:
import risk_first.pipeline as _pipeline_mod
import risk_first.training.ppo  as _ppo_mod
import risk_first.training.cppo as _cppo_mod

_ppo_mod.ppo_train   = ppo_train_gpu
_cppo_mod.cppo_train = cppo_train_gpu
_pipeline_mod.ppo_train  = ppo_train_gpu
_pipeline_mod.cppo_train = cppo_train_gpu

print("Trainers patched to GPU versions.")

## 6. Download HuggingFace data (once)

In [ ]:
from pathlib import Path
from risk_first.data.load_hf import load_datasets as load_hf_datasets

cache_dir = Path("risk_first/cache")
if not (cache_dir / "train.csv").exists():
    print("Downloading benstaf/nasdaq_2013_2023 from HuggingFace...")
    load_hf_datasets(cache_dir=str(cache_dir))
    print("Download complete.")
else:
    print("Cache already exists, skipping download.")

## 7. Run MULTI-SEED Ablation A→F

> ⚠️ **This loops over multiple random seeds.** Results are stored continuously.

In [ ]:
import yaml
import pandas as pd
import numpy as np
import shutil
from pathlib import Path
from risk_first.run_ablation import run_ablation

SEEDS = [10, 20, 30, 40, 50]
all_summaries = []

with open("risk_first/config.yaml", "r") as f:
    base_cfg = yaml.safe_load(f)

for seed in SEEDS:
    print(f"\n{'='*40}")
    print(f"=== RUNNING ABLATION WITH SEED {seed:02d} ===")
    print(f"{'='*40}\n")
    
    base_cfg['seed'] = seed
    with open("risk_first/config.yaml", "w") as f:
        yaml.safe_dump(base_cfg, f)
        
    # Run all configs for this seed
    run_ablation(cfg_path="risk_first/config.yaml")
    
    summary = pd.read_csv("risk_first/results/ablation_summary.csv", index_col=0)
    summary['Seed'] = seed
    all_summaries.append(summary)
    
    # Save seed-specific equity curves to avoid overwrite
    seed_dir = Path(f"risk_first/results/seed_{seed}")
    seed_dir.mkdir(exist_ok=True, parents=True)
    for npy_file in Path("risk_first/results").glob("*.npy"):
        shutil.copy(npy_file, seed_dir / npy_file.name)

print("\n=== ALL SEEDS COMPLETE ===")

## 8. Aggregate Results & Display Final Table (Mean ± Std)

In [ ]:
full_results = pd.concat(all_summaries)
full_results.to_csv("risk_first/results/multiseed_raw_results.csv")

numeric_cols = ['Cumulative Return', 'Sharpe Ratio', 'Rachev Ratio', 'Outperf. Freq.', 'Max Drawdown']

def clean_pct(x):
    if isinstance(x, str) and '%' in x:
        return float(x.replace('%', ''))
    return float(x)

for col in numeric_cols:
    if col in full_results.columns:
        full_results[col] = full_results[col].apply(clean_pct)

mean_results = full_results.groupby(full_results.index)[numeric_cols].mean()
std_results = full_results.groupby(full_results.index)[numeric_cols].std()

final_table = pd.DataFrame(index=mean_results.index)
for col in numeric_cols:
    if col in mean_results.columns:
        if 'Return' in col or 'Drawdown' in col or 'Freq' in col:
            final_table[col] = mean_results[col].map('{:.2f}%'.format) + " ± " + std_results[col].map('{:.2f}%'.format)
        else:
            final_table[col] = mean_results[col].map('{:.3f}'.format) + " ± " + std_results[col].map('{:.3f}'.format)

print("\n=== FINAL MULTI-SEED ABLATION TABLE (5 Seeds) ===\n")
print(final_table.to_string())
final_table.to_csv("risk_first/results/multiseed_final_table.csv")

## 9. Download all results as ZIP
Save to your Google Drive to ensure you don't lose the data.

In [ ]:
import shutil
from google.colab import drive

drive.mount('/content/drive')
shutil.make_archive("/content/drive/MyDrive/finrl_multiseed_results", "zip", "risk_first/results")
print("Results zipped and saved to Google Drive: /content/drive/MyDrive/finrl_multiseed_results.zip")